# Portfolio analytics

This notebook only orchestrates: it calls `transactions` (trade logic), `prices` (Yahoo Finance fetch + cache), `returns` (CAGR / HYSA benchmark math), and `visualization` (all charts). No logic lives in this notebook itself — see `docs/architecture.md` for the module map.

In [ ]:
from datetime import date

from trades import prices, returns, transactions, visualization
from trades.brokers.ibkr import main
from trades.config import AggregationConfig, IbkrFlexApiConfig, PriceApiConfig, ReturnsConfig

AS_OF_DATE = date.today()  # change this to price the portfolio as of any past date

# Every tunable parameter lives on one of these config objects (see
# docs/architecture.md#configuration) — nothing here is a hidden default.
aggregation_config = AggregationConfig()
price_api_config = PriceApiConfig()
returns_config = ReturnsConfig()
ibkr_config = IbkrFlexApiConfig()

## 1. Load, enrich, and aggregate trades

Trades come from the local IBKR ledger cache (`data/brokers/ibkr/ledger.csv`, kept fresh by running `notebooks/ibkr_sync.ipynb`), standardized from the canonical ledger onto the older, narrower trade schema by `preprocessing.standardize_ibkr_trades` (only `BUY` events on a real symbol — see `docs/architecture.md`, "Canonical trade schema"). `enrich_trades` then adds `usd_per_share`, and `aggregate_same_day_trades` merges same-day, same-symbol fills executed within 0.01% of each other into one row (summed shares/USD, recomputed $/share) — this is the dataset used for everything downstream.

In [ ]:
ledger = main.load_ledger(ibkr_config)
raw = preprocessing.standardize_ibkr_trades(ledger)
enriched = transactions.enrich_trades(raw)
trades = transactions.aggregate_same_day_trades(enriched, aggregation_config)
print(f"{len(raw)} standardized buys -> {len(trades)} aggregated trades")
trades

23 standardized buys -> 14 aggregated trades


,trade_date,symbol,shares,usd_spent,usd_per_share,n_trades
0,2026-01-27,VOO,0.1500,96.058500,640.390000,1
1,2026-04-10,BND,1.6950,124.997775,73.745000,1
2,2026-04-10,VOO,1.3974,874.968036,626.140000,1
3,2026-04-10,VXUS,3.0706,249.992899,81.415000,1
4,2026-05-05,BND,3.4129,249.994925,73.250000,2
5,2026-05-05,VOO,2.6281,1749.967847,665.868059,3
6,2026-05-05,VXUS,5.9858,499.991804,83.529654,2
7,2026-05-06,BND,0.0039,0.286650,73.500000,1
8,2026-06-01,QQQM,6.1700,1889.994400,306.320000,2
9,2026-06-01,VOO,9.4672,6612.933872,698.510000,2


## 2. Investment schedule

Totals per symbol, invested-per-month (overall and per symbol), the daily investment timeline (with gaps between buys), and a pie breakdown with a menu to switch between whole-portfolio-by-symbol and any one symbol's by-date split.

In [3]:
total_by_symbol = transactions.total_invested_by_symbol(trades)
print("Total invested to date, by symbol:")
print(total_by_symbol.to_string())
print(f"\nTotal invested to date, whole portfolio: ${total_by_symbol.sum():,.2f}")

Total invested to date, by symbol:
symbol
VOO     11333.881528
QQQM     1889.994400
VXUS     1652.433236
BND       376.157630

Total invested to date, whole portfolio: $15,252.47


In [4]:
monthly = transactions.monthly_invested(trades)
monthly

symbol,BND,QQQM,VOO,VXUS,Total
month,,,,,
2026-01,0.000000,0.0000,96.058500,0.000000,96.058500
2026-04,124.997775,0.0000,874.968036,249.992899,1249.958710
2026-05,250.281575,0.0000,1749.967847,499.991804,2500.241226
2026-06,0.878280,1889.9944,8612.887145,902.448533,11406.208358


In [5]:
visualization.plot_monthly_invested(monthly).show()

In [6]:
daily = transactions.daily_investment_timeline(trades)
visualization.plot_daily_investment_timeline(daily).show()

In [7]:
pie_options = transactions.pie_chart_options(trades)
visualization.plot_investment_pie(pie_options).show()

## 3. Price history

One local cache file per symbol (`data/prices/{SYMBOL}.csv`), each call only fetching the date range missing since the last run — see `docs/architecture.md` for the cache design. History goes back to the first trade date across the whole portfolio.

In [8]:
symbols = sorted(trades["symbol"].unique())
first_trade_date = trades["trade_date"].min().date()

price_histories = prices.update_price_caches(
    symbols, since=first_trade_date, as_of=AS_OF_DATE, config=price_api_config
)
for symbol, history in price_histories.items():
    latest_close = history["close"].iloc[-1]
    print(f"{symbol}: {len(history)} trading days cached, latest close {latest_close:.2f}")

BND: 109 trading days cached, latest close 73.11
QQQM: 109 trading days cached, latest close 293.42
VOO: 109 trading days cached, latest close 684.84
VXUS: 109 trading days cached, latest close 84.84


## 4. Returns vs. a HYSA benchmark

For each trade: current price, days held, total return, CAGR-style annualized return, the compounded HYSA return over the same window (rate set by `returns_config.hysa_annual_rate`, default 4%), and the resulting alpha. See `docs/returns.md` for the derivation of each step.

In [9]:
def price_lookup(symbol: str, as_of: date) -> float | None:
    return prices.price_as_of(price_histories[symbol], as_of)


returns_df = returns.build_returns_table(
    trades, price_lookup, as_of=AS_OF_DATE, config=returns_config
)

display_table = returns_df[
    [
        "trade_date",
        "symbol",
        "usd_per_share",
        "current_price",
        "days_held",
        "total_return_pct",
        "annualized_return_pct",
        "hysa_period_return_pct",
        "alpha_period_pct",
    ]
].rename(columns={"usd_per_share": "price_paid"})
display_table

,trade_date,symbol,price_paid,current_price,days_held,total_return_pct,annualized_return_pct,hysa_period_return_pct,alpha_period_pct
0,2026-01-27,VOO,640.390000,684.840027,157,6.941087,16.884397,1.701339,5.239748
1,2026-04-10,BND,73.745000,73.110001,84,-0.861074,-3.688047,0.906700,-1.767774
2,2026-04-10,VOO,626.140000,684.840027,84,9.374904,47.606885,0.906700,8.468205
3,2026-04-10,VXUS,81.415000,84.839996,84,4.206837,19.608849,0.906700,3.300137
4,2026-05-05,BND,73.250000,73.110001,59,-0.191125,-1.176542,0.635993,-0.827118
5,2026-05-05,VOO,665.868059,684.840027,59,2.849208,18.981782,0.635993,2.213216
6,2026-05-05,VXUS,83.529654,84.839996,59,1.568715,10.108305,0.635993,0.932722
7,2026-05-06,BND,73.500000,73.110001,58,-0.530611,-3.292655,0.625179,-1.155791
8,2026-06-01,QQQM,306.320000,293.420013,32,-4.211278,-38.783691,0.344445,-4.555723
9,2026-06-01,VOO,698.510000,684.840027,32,-1.957019,-20.183158,0.344445,-2.301464


In [10]:
numeric_cols = display_table.select_dtypes("number").columns
display_rounded = display_table.assign(**{c: display_table[c].round(2) for c in numeric_cols})
visualization.render_table(display_rounded, title=f"Returns as of {AS_OF_DATE}").show()

portfolio_alpha = returns.portfolio_alpha_pct(returns_df)
label = (
    f"Dollar-weighted portfolio alpha vs. {returns_config.hysa_annual_rate:.0%} HYSA "
    "(period, not annualized)"
)
print(f"{label}: {portfolio_alpha:+.2f}%")

Dollar-weighted portfolio alpha vs. 4% HYSA (period, not annualized): -0.70%


## 5. Annualized return curve

Per-trade annualized return against days held, with a fitted trend and the flat HYSA benchmark line. Short holds annualize into large, noisy numbers by design — that's why the combined alpha above uses period alpha instead of this annualized figure.

In [11]:
trend_x, trend_y = returns.fit_trend(
    returns_df["days_held"].to_numpy(),
    returns_df["annualized_return_pct"].to_numpy(),
    returns_config,
)
visualization.plot_return_curve(returns_df, trend_x, trend_y, returns_config).show()